# Lesson 05 — Exercise Solutions

Worked solutions with a short rationale after each. Run setup.sql first.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

How many products exist in each product category? Join through ProductSubcategory to get to ProductCategory.
Expected columns: CategoryName, ProductCount
Tables: Production.Product, Production.ProductSubcategory, Production.ProductCategory

In [ ]:
SELECT
    pc.Name  AS CategoryName,
    COUNT(*) AS ProductCount
FROM Production.Product              AS p
JOIN Production.ProductSubcategory  AS ps ON ps.ProductSubcategoryID = p.ProductSubcategoryID
JOIN Production.ProductCategory     AS pc ON pc.ProductCategoryID    = ps.ProductCategoryID
GROUP BY pc.Name
ORDER BY ProductCount DESC;

**Why:** Two-hop join: Product → ProductSubcategory → ProductCategory, then GROUP BY.

## Exercise 2

Find the top 3 sales territories by total revenue in 2013.
Expected columns: TerritoryID, TotalRevenue
Table: Sales.SalesOrderHeader

In [ ]:
SELECT TOP 3
    TerritoryID,
    SUM(TotalDue) AS TotalRevenue
FROM Sales.SalesOrderHeader
WHERE YEAR(OrderDate) = 2013
GROUP BY TerritoryID
ORDER BY TotalRevenue DESC;

**Why:** Filter year in WHERE, GROUP BY territory, ORDER BY + TOP.

## Exercise 3

Find customers who placed more than 10 orders AND whose average order value exceeded $2,000.
Expected columns: CustomerID, OrderCount, AvgOrderValue

In [ ]:
SELECT
    CustomerID,
    COUNT(*)        AS OrderCount,
    AVG(TotalDue)   AS AvgOrderValue
FROM Sales.SalesOrderHeader
GROUP BY CustomerID
HAVING COUNT(*) > 10
   AND AVG(TotalDue) > 2000
ORDER BY AvgOrderValue DESC;

**Why:** Both conditions go in HAVING — they operate on aggregated data.

## Exercise 4

Using ROLLUP, show total sales by year and by overall grand total. Only include years 2012–2014.
Expected columns: OrderYear, TotalSales
(Grand total row will have NULL OrderYear)

In [ ]:
SELECT
    YEAR(OrderDate) AS OrderYear,
    SUM(TotalDue)   AS TotalSales
FROM Sales.SalesOrderHeader
WHERE YEAR(OrderDate) BETWEEN 2012 AND 2014
GROUP BY ROLLUP (YEAR(OrderDate))
ORDER BY OrderYear;

**Why:** Filter in WHERE first; ROLLUP adds year subtotals and a grand total (NULL year).

## Exercise 5

For each product subcategory, show the most expensive product's ListPrice and the average ListPrice. Include only subcategories with avg ListPrice > $200.
Expected columns: SubcategoryName, MaxPrice, AvgPrice

In [ ]:
SELECT
    ps.Name        AS SubcategoryName,
    MAX(p.ListPrice) AS MaxPrice,
    AVG(p.ListPrice) AS AvgPrice
FROM Production.Product            AS p
JOIN Production.ProductSubcategory AS ps ON ps.ProductSubcategoryID = p.ProductSubcategoryID
WHERE p.ListPrice > 0
GROUP BY ps.Name
HAVING AVG(p.ListPrice) > 200
ORDER BY AvgPrice DESC;

**Why:** JOIN subcategory, GROUP BY, HAVING on the average.

## Challenges

Harder problems. They may combine this lesson's topic with
earlier lessons, and usually need 2+ concepts in one query.

### Challenge 1

One result set with revenue at three levels — per (territory, year), per territory subtotal, and grand total — using GROUPING SETS, with a readable Level column ('Detail'/'Territory total'/'Grand total') derived from GROUPING_ID.
Expected columns: TerritoryName, OrderYear, Revenue, Level

In [ ]:
SELECT
    st.Name AS TerritoryName,
    YEAR(soh.OrderDate) AS OrderYear,
    SUM(soh.TotalDue) AS Revenue,
    CASE GROUPING_ID(st.Name, YEAR(soh.OrderDate))
        WHEN 0 THEN 'Detail'
        WHEN 1 THEN 'Territory total'
        WHEN 3 THEN 'Grand total'
    END AS Level
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesTerritory st ON st.TerritoryID = soh.TerritoryID
GROUP BY GROUPING SETS (
    (st.Name, YEAR(soh.OrderDate)),
    (st.Name),
    ()
)
ORDER BY GROUPING(st.Name), st.Name, GROUPING(YEAR(soh.OrderDate)), OrderYear;

**Why:** GROUPING SETS ((Territory,Year),(Territory),()) yields exactly the three
levels needed in one pass, no UNION of three queries. GROUPING_ID packs
both columns' grouping bits into an int (0=detail, 1=territory subtotal,
3=grand total) that CASE turns into a readable label. Tempting-but-wrong:
ROLLUP(Territory,Year) looks similar but only rolls up in declared column
order, so it can't express arbitrary/independent grouping combinations.

### Challenge 2

The correct "average order value" per territory: SUM(TotalDue)/COUNT(DISTINCT SalesOrderID), shown next to the naive AVG(TotalDue) over a header+detail join, with a comment explaining why they differ.
Expected columns: TerritoryName, NaiveAvg, CorrectAvg
Hint: The join fans out headers; AVG then weights big orders by line count.

In [ ]:
SELECT
    n.TerritoryName,
    n.NaiveAvg,
    c.CorrectAvg
FROM (
    SELECT st.Name AS TerritoryName, AVG(soh.TotalDue) AS NaiveAvg
    FROM Sales.SalesOrderHeader soh
    JOIN Sales.SalesOrderDetail sod ON sod.SalesOrderID = soh.SalesOrderID
    JOIN Sales.SalesTerritory st ON st.TerritoryID = soh.TerritoryID
    GROUP BY st.Name
) AS n
JOIN (
    SELECT st.Name AS TerritoryName,
           SUM(soh.TotalDue) / COUNT(DISTINCT soh.SalesOrderID) AS CorrectAvg
    FROM Sales.SalesOrderHeader soh
    JOIN Sales.SalesTerritory st ON st.TerritoryID = soh.TerritoryID
    GROUP BY st.Name
) AS c ON c.TerritoryName = n.TerritoryName
ORDER BY n.TerritoryName;

**Why:** The naive column joins header to detail (to reproduce the bug) and AVGs
TotalDue over the fanned rows, so multi-line orders pull the average up.
Correct = aggregate SalesOrderHeader alone (no detail join) in its own
subquery: SUM(TotalDue)/COUNT(DISTINCT SalesOrderID) = real revenue over
real order count. Tempting-but-wrong: applying that same formula to the
fanned join still inflates the numerator even with a DISTINCT denominator.

### Challenge 3

A matrix of order counts: one row per territory, one column per year 2011–2014, built with conditional aggregation (SUM(CASE...)).
Expected columns: TerritoryName, Y2011, Y2012, Y2013, Y2014

In [ ]:
SELECT
    st.Name AS TerritoryName,
    SUM(CASE WHEN YEAR(soh.OrderDate) = 2011 THEN 1 ELSE 0 END) AS Y2011,
    SUM(CASE WHEN YEAR(soh.OrderDate) = 2012 THEN 1 ELSE 0 END) AS Y2012,
    SUM(CASE WHEN YEAR(soh.OrderDate) = 2013 THEN 1 ELSE 0 END) AS Y2013,
    SUM(CASE WHEN YEAR(soh.OrderDate) = 2014 THEN 1 ELSE 0 END) AS Y2014
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesTerritory st ON st.TerritoryID = soh.TerritoryID
GROUP BY st.Name
ORDER BY st.Name;

**Why:** One row per territory with one column per year means the year has to move
from rows into columns — conditional aggregation does that directly: one
SUM(CASE WHEN YEAR(OrderDate)=<year> THEN 1 ELSE 0 END) per year, all four
computed in a single GROUP BY pass over the table. Tempting-but-wrong: four
separate per-year queries UNIONed together would need a second pivot step
afterwards and scan the table four times instead of once.

### Challenge 4

Territories whose 2013 revenue exceeded their 2012 revenue by more than 10% — aggregate per territory-year first, then compare years. Decide (and justify in the comment) whether the comparison belongs in HAVING or in an outer query.
Expected columns: TerritoryName, Rev2012, Rev2013, GrowthPct
Hint: Conditional aggregation puts both years on one row.

In [ ]:
WITH TerritoryYear AS (
    SELECT
        st.Name AS TerritoryName,
        SUM(CASE WHEN YEAR(soh.OrderDate) = 2012 THEN soh.TotalDue ELSE 0 END) AS Rev2012,
        SUM(CASE WHEN YEAR(soh.OrderDate) = 2013 THEN soh.TotalDue ELSE 0 END) AS Rev2013
    FROM Sales.SalesOrderHeader soh
    JOIN Sales.SalesTerritory st ON st.TerritoryID = soh.TerritoryID
    GROUP BY st.Name
)
SELECT
    TerritoryName,
    Rev2012,
    Rev2013,
    ROUND((Rev2013 - Rev2012) * 100.0 / Rev2012, 2) AS GrowthPct
FROM TerritoryYear
WHERE Rev2012 > 0
  AND (Rev2013 - Rev2012) * 100.0 / Rev2012 > 10
ORDER BY GrowthPct DESC;

**Why:** Conditional aggregation puts each territory's 2012 and 2013 revenue on one
row (a CTE). The growth comparison goes in an OUTER WHERE, not HAVING:
T-SQL's HAVING cannot reference SELECT-list aliases like Rev2012/Rev2013
(verified — "Invalid column name"), so a HAVING version would repeat the
full CASE expressions and division a second time. The CTE computes each
aggregate once and reuses it for both filtering and ORDER BY.